**What is Auto Loade**r? is a cloud-native tool in Databricks that incrementally and efficiently processes new data files as they arrive in cloud storage.How It WorksUses a Spark Structured Streaming source called cloudFiles.Monitors a cloud storage directory path to process new files automatically.Tracks files using a scalable key-value store (RocksDB) in the checkpoint location for exactly-once processing guarantees

In [0]:
#%sql
#use dbacademy.bronze;  -- SQL code
#spark.sql("USE dbacademy.bronze")
spark.catalog.setCurrentCatalog("dbacademy")
spark.catalog.setCurrentDatabase("bronze")

In [0]:

df = (spark.readStream.format("cloudFiles")
.option("cloudFiles.format", "csv")
.option("cloudFiles.schemaLocation", "/Volumes/dbacademy/bronze/autoloader/Input/schema/")
.option("cloudFiles.inferSchema", "true")
.option("cloudFiles.schemaEvolutionMode", "rescue")
.load("/Volumes/dbacademy/bronze/autoloader/Input/")
)

In [0]:
(df.writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", "/Volumes/dbacademy/bronze/autoloader/output/checkpoint/")
  .trigger(availableNow=True)
  .start("/Volumes/dbacademy/bronze/autoloader/output/")
)

In [0]:
df1= spark.read.format("delta").load("/Volumes/dbacademy/bronze/autoloader/output/");
display(df1)